In [73]:
%load_ext pretty_jupyter

# Introduction

This walk-through is part of the [ECLR](https://datasquad.github.io/ECLR/) page.

In this walkthrough you will learn to 

* use SQL to access database
* ... 

You have been studying Python such that you can do simple or very complex dat analysis. Why would you need to know SQL (which stands for Structured Query Language)? These days all large (and some small) businesses keep enourmous amounts of data (think every interaction you as a customer may have with the Amazon webpage). These data are saved in whgat is called databases. Databases are really collections of spreadsheets (with observations in rows and variables in columns). Often the data is organised such that different spreadsheets can potentially be linked as they have common keys. Such databases are called relational databases.

There may be a lot of interesting analysis you could do with some of the data in such a database. However, these databases are often so big that you will have to first extract some suitable subset of the data (perhaps already aggregated to a certain level) such that the dataset's structure and size is suitable for analysis in Python.

The language used to access and extract data is SQL and it stands at the beginning of many data projects.

# Database setup

Unsurprisingly databases come in different formats. However, the good thing is that communicating with these always happens with variants of the SQL language. For certain applications and database formats you can access databases directly from Python. This is what we will cover here, we assume that a database is stored locally and then we use Python to communicate with that database. 

In commercial organisations, where datbases are accessed by many people independently and simultaneously, this access needs to be controlled by a database server. But here we will abstract from such difficulties. When you work in a particular organisation you will be introduced to any such specifics.

We use an example database that has been created for teaching purposes called `northwind.db`. This is built as a SQLlite type database which can just be saved locally on your computer without the need for a server. As such in can then be directly accessed from, say, Python.

Go to [JP White's github page](https://github.com/jpwhite3/northwind-SQLite3) and download the `northwind.db` database file and save it into your data directory (or directory you are working from).

Next we install libraries we will be using in this workthrough. Importantly, one of them is the `sqlite3` library which will allow us to communicate with the SQLite formatted database `northwind.db`.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
from lets_plot import *
import statsmodels.api as sm 

# Lets_plot requires a setup
LetsPlot.setup_html(no_js=True)

Before we continue we shall test whether we can connect to the database. Run the following command and you should get the following output.

In [2]:
con = sqlite3.connect("../data/northwind.db")
pd.read_sql_query("SELECT COUNT(*) AS n FROM Orders;", con)


,n
0,16282


Without talking about the detail of this query, this output tells you that there are more than 16K rows of data in the `Orders` spreadsheet.

The only coding aspect we need to take from this is how the SQL code to communicate with the database is delivered. It is delivered as a text command inside the `pd.read_sql_query` function. The second input into the function is `con` which was defined above. `con` is the connection to the database and we will need it whenever we communicate with the database.

```python
pd.read_sql_query("SQL code finished with a ;", con)
```

# Exploring the database

A database will often have many different tables. Think an EXCEL file with multiple sheets. When working with a database it is important to understand the structure of the database, meaning, which tables does it contain and which variables are in each table. Understanding which variables are contained in each table is important so that we can link data from different tables.

You can see a structure of the database on [JP White's github page for the northwind database](https://github.com/jpwhite3/northwind-SQLite3).

## Identify tables

We use the following command to get the names of the tables contained in the Nortwind database.

In [3]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con)

,name
0,Categories
1,sqlite_sequence
2,CustomerCustomerDemo
3,CustomerDemographics
4,Customers
5,Employees
6,EmployeeTerritories
7,Order Details
8,Orders
9,Products




---
#### Finding code
[//]: # (-.- .tabset)

##### Websearch

As we are at the beginning of our SQL learning journey you cannot be expected to know how to get a list of the tables from the database. You could use your favourite search engine and search for something like "How to use SQL to get all table names in a database sqlite". In my case this led me to the following website [that provides quite clear but wordy instruction on how to do that](https://www.datacamp.com/tutorial/sqlite-show-tables).

This will give the SQL command that needs to be inserted into the `pd.read_sql_query` function.

##### AI

You could also ask your favourite AI engine. But you will have to give the AI enough information. For instance you could ask the following: "I am working in Python and am accessing a SQLite database called "northwind.db". How can I get a list of all tables in the database?"

#### 
[//]: # (-.- .unlisted .unnumbered)

---

This database has 13 tables. It represents the database for a small company with customers, inventory, purchasing, suppliers, employees etc. 

## Accessing data from a table

Let's say we want to access data from the `Suppliers` table. It will be important to know what variables are included in this table.


In [4]:
pd.read_sql_query("PRAGMA table_info(Suppliers);", con)

,cid,name,type,notnull,dflt_value,pk
0,0,SupplierID,INTEGER,1,None,1
1,1,CompanyName,TEXT,1,None,0
2,2,ContactName,TEXT,0,None,0
3,3,ContactTitle,TEXT,0,None,0
4,4,Address,TEXT,0,None,0
5,5,City,TEXT,0,None,0
6,6,Region,TEXT,0,None,0
7,7,PostalCode,TEXT,0,None,0
8,8,Country,TEXT,0,None,0
9,9,Phone,TEXT,0,None,0



---
#### What is PRAGMA
[//]: # (-.- .tabset)

##### SQLite specific solution

The above code using the PRAGMA command is a solution to the particular problem at hand using a SQLite specific command the PRAGMA command. It makes certain things much easier than using SQL language that is universal across different database formats. But is if your database is not in the SQLite format, then this will not be available.

##### A generic SQL solution

If you want to get a list of variables in a table in a way that should be applicable across different database formats you do the following:


In [5]:
cursor = con.cursor()
cursor.execute("SELECT * FROM Suppliers LIMIT 0;")
columns = [desc[0] for desc in cursor.description]
print(columns)


['SupplierID', 'CompanyName', 'ContactName', 'ContactTitle', 'Address', 'City', 'Region', 'PostalCode', 'Country', 'Phone', 'Fax', 'HomePage']


As you can see this merely prints a list variables and we will not go through the details of this command here.

The reality is that you may be able to find easier solutions specific to your particular database format.

#### 
[//]: # (-.- .unlisted .unnumbered)

---

Now we may wish to look at the `Suppliers` database and find all the suppliers that are from the UK. Let's see how this works and then we will disect the command (which is again delivered via the `pd.read_sql_query` function). 

In [6]:
pd.read_sql_query("SELECT CompanyName, City FROM Suppliers WHERE Country = 'UK' \
                  ORDER BY CompanyName;", con)

,CompanyName,City
0,Exotic Liquids,London
1,"Specialty Biscuits, Ltd.",Manchester


First, note that the line break is indicated by "\". It is important that there is no space after the "\".

If you wish to save the results of this query into a spreadsheet you can do this as follows. First save the query in a DataFrame and then save that using the `to_csv` method.

In [9]:
uk_suppliers = pd.read_sql_query("SELECT CompanyName, City FROM Suppliers WHERE Country = 'UK' \
                  ORDER BY CompanyName;", con)
uk_suppliers.to_csv('uk_suppliers.csv', index=False)


When looking at the result it becomes obvious what the above SQL command did.

```python
# selects variables CompanyName and City from the Suppliers table
SELECT CompanyName, City FROM Suppliers
# but only the rows where country is equal to UK
WHERE Country = 'UK' 
# Then order the result by CompanyName
ORDER BY CompanyName;
```


---
#### Practice Exercise
[//]: # (-.- .tabset)

##### Task

Find out of which products the company has more than 100 units in stock (`UnitsInStock > 100`). Create a table with these products and include the following variables into the table: `ProductName`, `QuantityPerUnit`, `MinOrderAmount`, `UnitsInStock`. But check whether all these variables actually exist in the `Products` table.

##### Solution Strategy

First check which of the above four variables actually do exist. Adjust the earlier `PRAGMA` command to achieve this.

Then, you should adjust the above `SELECT ... FROM ... WHERE` command to the question at hand.

##### Solution

First check which variables are available in `Products`.

In [7]:
pd.read_sql_query("PRAGMA table_info(Products);", con)

,cid,name,type,notnull,dflt_value,pk
0,0,ProductID,INTEGER,1,None,1
1,1,ProductName,TEXT,1,None,0
2,2,SupplierID,INTEGER,0,None,0
3,3,CategoryID,INTEGER,0,None,0
4,4,QuantityPerUnit,TEXT,0,None,0
5,5,UnitPrice,NUMERIC,0,0,0
6,6,UnitsInStock,INTEGER,0,0,0
7,7,UnitsOnOrder,INTEGER,0,0,0
8,8,ReorderLevel,INTEGER,0,0,0
9,9,Discontinued,TEXT,1,'0',0


Now we know that only `ProductName`, `QuantityPerUnit` and `UnitsInStock` do exist. The following command selects these three variables, using the condition `WHERE UnitsInStock > 100`.

In [8]:
pd.read_sql_query("SELECT ProductName, QuantityPerUnit, UnitsInStock FROM Products \
                  WHERE UnitsInStock > 100 \
                  ORDER BY UnitsInStock;", con)

,ProductName,QuantityPerUnit,UnitsInStock
0,Röd Kaviar,24 - 150 g jars,101
1,Gustaf's Knäckebröd,24 - 500 g pkgs.,104
2,Sasquatch Ale,24 - 12 oz bottles,111
3,Geitost,500 g,112
4,Inlagd Sill,24 - 250 g jars,112
5,Sirop d'érable,24 - 500 ml bottles,113
6,Pâté chinois,24 boxes x 2 pies,115
7,Grandma's Boysenberry Spread,12 - 8 oz jars,120
8,Boston Crab Meat,24 - 4 oz tins,123
9,Rhönbräu Klosterbier,24 - 0.5 l bottles,125



#### 
[//]: # (-.- .unlisted .unnumbered)

---

# Grouping and Aggregating Data

Often you will find that a database contains information on individual customers or individual transactions. The amount of data can be very large and for an analysis you may not be interested in the individual observations but rather in aggregated information. If so, then it may not be necessary to download the individual observations but it would be possible to aggregate information, for instance to a daily transaction summaries, which can then be downloaded for analysis.

The first task we will undertake is to summarise the number of orders (from the `Orders` table) that go to different countries.

---
#### Understand the structure of `Orders`
[//]: # (-.- .tabset)

##### Question

Before proceeding with this you will want to check out what variables are included in the `Orders` table and in particular what the variable is called that contains the country information. Attempt to apply the previously introduced `PRAGMA table_info(Products);` command (recall that this is convenient but specific to SQLite databases) to find the names of the variables.

##### Solution



In [12]:
pd.read_sql_query("PRAGMA table_info(Orders);", con)

,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,INTEGER,1,None,1
1,1,CustomerID,TEXT,0,None,0
2,2,EmployeeID,INTEGER,0,None,0
3,3,OrderDate,DATETIME,0,None,0
4,4,RequiredDate,DATETIME,0,None,0
5,5,ShippedDate,DATETIME,0,None,0
6,6,ShipVia,INTEGER,0,None,0
7,7,Freight,NUMERIC,0,0,0
8,8,ShipName,TEXT,0,None,0
9,9,ShipAddress,TEXT,0,None,0



From here you learn that the variable we are interested in is called `ShipCountry`

#### 
[//]: # (-.- .unlisted .unnumbered)

---


Now that we know the variable name of the country variable we can look at how many orders there are. Let's, for a moment, imagine you had a dataframe in Python called `Orders` and that had a variable called `ShipCountry`. How would achieve this in Python?

``` python
Orders.groupby('ShipCountry').size()
```

So there were two elements, we needed to group the data by the `ShipCountry` variable and then count, which here was achieved by using the `size()` method which does exactly count the number of observations in each group. Similar tasks have to be achieved when using a SQL query to the database.

Here is the command used:

In [22]:
pd.read_sql_query("SELECT ShipCountry, COUNT(*) AS n_orders FROM Orders GROUP BY ShipCountry ORDER BY n_orders DESC;", con)

,ShipCountry,n_orders
0,USA,2328
1,Germany,2193
2,France,1778
3,Brazil,1683
4,UK,1280
5,Mexico,899
6,Venezuela,707
7,Spain,691
8,Canada,547
9,Italy,538



---
#### What does this command do?
[//]: # (-.- .tabset)

##### Question

We want to disect this command as it helps us understand how SQL works. Can you identify the equivalents to `groupby` and `size()`?

##### Solution

Here is a detailed breakdown:

``` SQL
# get the ShipCountry column from Orders and COUNT the rows
SELECT ShipCountry, COUNT(*) AS n_orders FROM Orders 
```

You can try and run this command alone and see what happens. Basically you will have selected one variable and then you counted all the rows in the table. At this stage you have counted but 

``` SQL
GROUP BY ShipCountry 
```



``` SQL
ORDER BY n_orders DESC;
```


#### 
[//]: # (-.- .unlisted .unnumbered)

---



# Reading

* Database used here `northwind.db` is from [JP White's github page](https://github.com/jpwhite3/northwind-SQLite3)

* [Real Python intro to SQL](https://realpython.com/learning-paths/database-access-in-python/)